In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class Foldover(Scene):
    """Ring modulation's four sidebands as the modulator climbs past the
    carrier. The lower pair slides through zero, turns negative, and each one
    comes back up on the other side: the spectrum folds at 0 Hz, and what we
    hear is the mirror."""

    def construct(self):
        # the sideband purple of the chapter's text and figures, in the shade
        # the text uses on a light or a dark page
        dark = config.background_color.to_hex().upper() != "#FFFFFF"
        PURPLE = "#B39DDB" if dark else "#6E3B87"
        FC = 220.0            # carrier, fixed
        F_LO, F_HI = 40.0, 420.0
        SPAN = 700.0          # half-width of the frequency axis, in Hz
        H = 0.9               # stem height (all four sidebands share amplitude)

        fm = ValueTracker(F_LO)

        # the full spectrum, negative frequencies included
        top = NumberLine(x_range=[-SPAN, SPAN, 100], length=12.0,
                         color=anim.IRON, stroke_width=2,
                         include_ticks=False, include_numbers=False)
        top.move_to([0.0, 1.0, 0.0])
        zero_tick = Line([0, 0.85, 0], [0, 1.15, 0], color=anim.IRON,
                         stroke_width=2)
        zero_label = MathTex("0").scale(0.6).move_to([0.0, 0.55, 0.0])
        freq_label = Text("frequency", font_size=20)
        freq_label.move_to([5.1, 0.55, 0.0])

        # what the ear gets: the positive half, at the same scale
        bottom = NumberLine(x_range=[0, SPAN, 100], length=6.0,
                            color=anim.IRON, stroke_width=2,
                            include_ticks=False, include_numbers=False)
        bottom.move_to([3.0, -2.4, 0.0])
        heard_label = Text("what you hear", font_size=22)
        heard_label.move_to([-3.1, -2.4, 0.0])

        crease = DashedLine([0, -3.0, 0], [0, 2.3, 0], color=anim.STEEL,
                            stroke_width=2, dash_length=0.12)

        # the carrier, marked on both sides
        def carrier_tick(sign):
            p = top.n2p(sign * FC)
            return Line(p + DOWN * 0.16, p + UP * 0.16, color=anim.RED,
                        stroke_width=4)

        fc_ticks = VGroup(carrier_tick(1.0), carrier_tick(-1.0))
        fc_label = MathTex("f_c", color=anim.RED).scale(0.65)
        fc_label.move_to(top.n2p(FC) + UP * 1.2)
        fc_neg = MathTex("-f_c", color=anim.RED).scale(0.65)
        fc_neg.move_to(top.n2p(-FC) + UP * 1.2)

        # the modulator, marked on both sides like the carrier, riding the axis
        def modulator_ticks():
            g = VGroup()
            for sign in (1.0, -1.0):
                p = top.n2p(sign * fm.get_value())
                g.add(Line(p + DOWN * 0.16, p + UP * 0.16, color=anim.BLUE,
                           stroke_width=4))
            return g

        fm_ticks = always_redraw(modulator_ticks)
        fm_label = MathTex("f_m", color=anim.BLUE).scale(0.65)
        fm_label.add_updater(lambda m: m.move_to(
            top.n2p(fm.get_value()) + DOWN * 0.72))
        fm_neg = MathTex("-f_m", color=anim.BLUE).scale(0.65)
        fm_neg.add_updater(lambda m: m.move_to(
            top.n2p(-fm.get_value()) + DOWN * 0.72))

        def stem(axis, freq_fn, color, width=5):
            def make():
                p = axis.n2p(freq_fn())
                return Line(p, p + UP * H, color=color, stroke_width=width)
            return always_redraw(make)

        # upper pair: f_c + f_m and its mirror. lower pair: f_c - f_m and its
        # mirror, the two that fold
        upper = VGroup(
            stem(top, lambda: FC + fm.get_value(), PURPLE),
            stem(top, lambda: -(FC + fm.get_value()), PURPLE),
        )
        lower = VGroup(
            stem(top, lambda: FC - fm.get_value(), PURPLE),
            stem(top, lambda: fm.get_value() - FC, PURPLE),
        )
        heard = VGroup(
            stem(bottom, lambda: FC + fm.get_value(), PURPLE),
            stem(bottom, lambda: abs(FC - fm.get_value()), PURPLE),
        )

        # the running numbers
        fm_read = MathTex("f_m =").scale(0.6).move_to([-5.3, 3.2, 0.0])
        fm_val = DecimalNumber(F_LO, num_decimal_places=0, color=anim.BLUE)
        fm_val.scale(0.6).next_to(fm_read, RIGHT, buff=0.15)
        fm_val.add_updater(lambda m: m.set_value(fm.get_value()))
        low_read = MathTex("f_c - f_m =").scale(0.6)
        low_read.move_to([-5.0, 2.65, 0.0])
        low_val = DecimalNumber(FC - F_LO, num_decimal_places=0,
                                include_sign=True, color=PURPLE)
        low_val.scale(0.6).next_to(low_read, RIGHT, buff=0.15)
        low_val.add_updater(lambda m: m.set_value(FC - fm.get_value()))

        cross = MathTex("f_c - f_m = 0", color=PURPLE).scale(0.7)
        cross.move_to([0.0, -0.75, 0.0])
        mirror = MathTex("|f_c - f_m|", color=PURPLE).scale(0.65)
        mirror.move_to(bottom.n2p(abs(FC - F_HI)) + DOWN * 0.45)

        self.add(top, zero_tick, zero_label, freq_label, crease, fc_ticks,
                 fc_label, fc_neg, fm_ticks, fm_label, fm_neg, upper, lower,
                 fm_read, fm_val, low_read, low_val)
        self.play(FadeIn(top), FadeIn(crease), FadeIn(zero_tick),
                  FadeIn(zero_label), FadeIn(freq_label), FadeIn(fc_ticks),
                  FadeIn(fc_label), FadeIn(fc_neg), run_time=0.8)
        self.play(FadeIn(fm_ticks), FadeIn(fm_label), FadeIn(fm_neg), FadeIn(upper),
                  FadeIn(lower), FadeIn(fm_read), FadeIn(fm_val),
                  FadeIn(low_read), FadeIn(low_val), run_time=0.7)
        self.add(bottom, heard_label, heard)
        self.play(FadeIn(bottom), FadeIn(heard_label), FadeIn(heard),
                  run_time=0.6)
        self.wait(0.6)

        # the modulator climbs to meet the carrier: both gold stems reach zero
        self.play(fm.animate(rate_func=linear).set_value(FC), run_time=4.5)
        self.play(FadeIn(cross), run_time=0.5)
        self.wait(1.0)
        self.play(FadeOut(cross), run_time=0.4)

        # past the carrier: the lower sideband is negative, and its mirror
        # is what reaches the ear
        self.play(fm.animate(rate_func=linear).set_value(F_HI), run_time=3.6)
        self.play(FadeIn(mirror), run_time=0.5)
        self.wait(1.4)


anim.show(Foldover)